## 查看是否开仓的盈亏详情

In [3]:
import pandas as pd
from typing import Optional

def _make_pred_filter_desc(lower: Optional[float], upper: Optional[float]) -> str:
    if lower is not None and upper is not None:
        return f"({lower} < predicted_return < {upper})"
    if lower is not None:
        return f"(predicted_return > {lower})"
    if upper is not None:
        return f"(predicted_return < {upper})"
    return "(no predicted_return filter)"

def _apply_pred_filter(df: pd.DataFrame, lower: Optional[float], upper: Optional[float]) -> pd.DataFrame:
    s = df["predicted_return"]
    if lower is not None and upper is not None:
        return df[(s > lower) & (s < upper)].copy()
    if lower is not None:
        return df[s > lower].copy()
    if upper is not None:
        return df[s < upper].copy()
    return df.copy()

def _apply_date_window_trades(df: pd.DataFrame,
                              start_date: Optional[str],
                              end_date: Optional[str]) -> pd.DataFrame:
    """对 trades_ledger（按 entry_date）应用时间窗口"""
    out = df.copy()
    if start_date:
        out = out[out["entry_date"] >= pd.to_datetime(start_date)]
    if end_date:
        out = out[out["entry_date"] <= pd.to_datetime(end_date)]
    return out

def _apply_date_window_all(df: pd.DataFrame,
                           start_date: Optional[str],
                           end_date: Optional[str]) -> pd.DataFrame:
    """对 all_signals_trades（按 signal_date）应用时间窗口"""
    out = df.copy()
    if "signal_date" not in out.columns:
        # 兼容场景：有的导出也可能只有 entry_date
        if "entry_date" in out.columns:
            out = out.rename(columns={"entry_date": "signal_date"})
        else:
            raise KeyError("all_signals_trades 缺少 signal_date / entry_date 列")
    if start_date:
        out = out[out["signal_date"] >= pd.to_datetime(start_date)]
    if end_date:
        out = out[out["signal_date"] <= pd.to_datetime(end_date)]
    return out

def analyze_trades_aligned(
    trades_path: str,
    all_trades_path: str,
    pred_lower: Optional[float] = 0.1,
    pred_upper: Optional[float] = None,
    start_date: Optional[str] = None,   # 新增：起始日期（含）
    end_date: Optional[str] = None,     # 新增：结束日期（含）
):
    """
    对齐键：
      - trades_ledger: (code, strategy, entry_date)
      - all_signals_trades: (code, strategy, signal_date) ——> 对齐时改名为 entry_date

    过滤：
      - 预测阈值：pred_lower / pred_upper（下上界任意组合）
      - 时间窗口：trades 用 entry_date；all_trades 用 signal_date

    统计（过滤后）：
      1) trades_ledger / all_signals_trades：记录数、pnl均值、盈利数/占比（整体+按 strategy）
      2) 交集/差集（all - trades，对齐键后）：规模与差集的 pnl 统计（整体+按 strategy）
    """
    # ------- 读取 -------
    trades = pd.read_csv(trades_path, parse_dates=['entry_date'])
    all_trades = pd.read_csv(all_trades_path, parse_dates=['entry_date','signal_date'])

    # 列名统一
    for df in [trades, all_trades]:
        if 'code' not in df.columns and 'symbol' in df.columns:
            df['code'] = df['symbol'].astype(str)
        else:
            df['code'] = df['code'].astype(str)

    # ------- 时间窗口过滤 -------
    trades = _apply_date_window_trades(trades, start_date, end_date)
    all_trades = _apply_date_window_all(all_trades, start_date, end_date)

    # ------- 预测阈值过滤 -------
    cond_desc = _make_pred_filter_desc(pred_lower, pred_upper)
    t_sel = _apply_pred_filter(trades, pred_lower, pred_upper)
    a_sel = _apply_pred_filter(all_trades, pred_lower, pred_upper)

    # ------- 输出函数：整体 + 按 strategy -------
    def summarize(df: pd.DataFrame, name: str):
        if df.empty:
            print(f"\n[{name}] 无记录 {cond_desc}"
                  + (f", 时间段[{start_date or '-inf'} ~ {end_date or '+inf'}]" if (start_date or end_date) else ""))
            return
        total = len(df)
        avg_pnl = df['pnl_pct'].mean()
        win_count = (df['pnl_pct'] > 0).sum()
        win_rate = win_count / total if total > 0 else 0.0
        print(f"\n[{name}] 条件{cond_desc}"
              + (f", 时间段[{start_date or '-inf'} ~ {end_date or '+inf'}]" if (start_date or end_date) else ""))
        print(f"总数={total}, pnl均值={avg_pnl:.4f}, 盈利数={win_count}, 盈利占比={win_rate:.2%}")
        by_strat = df.groupby('strategy')['pnl_pct'].agg(
            count='count',
            avg_pnl='mean',
            win_count=lambda s: (s > 0).sum(),
            win_rate=lambda s: (s > 0).mean()
        )
        print("按 strategy：")
        print(by_strat)

    # ------- 1) 分别统计 -------
    summarize(t_sel, "Trades Ledger")
    summarize(a_sel, "All Signals Trades")

    # ------- 2) 交集/差集（ledger.entry_date ↔ all.signal_date）-------
    ledger_keys = t_sel[['code', 'strategy', 'entry_date']].drop_duplicates()
    all_keys = a_sel[['code', 'strategy', 'signal_date']].drop_duplicates() \
                     .rename(columns={'signal_date': 'entry_date'})

    inter_keys = pd.merge(ledger_keys, all_keys, on=['code','strategy','entry_date'], how='inner')
    diff_keys = pd.merge(all_keys, ledger_keys,
                         on=['code','strategy','entry_date'],
                         how='left', indicator=True)
    diff_keys = diff_keys[diff_keys['_merge'] == 'left_only'].drop(columns=['_merge'])

    print(f"\n相同记录数: {len(inter_keys)}, 不同记录数(候选未开仓): {len(diff_keys)}  条件{cond_desc}"
          + (f", 时间段[{start_date or '-inf'} ~ {end_date or '+inf'}]" if (start_date or end_date) else ""))

    # 差集取完整行（注意把 entry_date -> signal_date 再对齐回去）
    if not diff_keys.empty:
        diff_keys_for_join = diff_keys.rename(columns={'entry_date': 'signal_date'})
        diff_trades = a_sel.merge(diff_keys_for_join,
                                  on=['code','strategy','signal_date'], how='inner')
    else:
        diff_trades = a_sel.iloc[0:0].copy()

    summarize(diff_trades, "差集 All_only（候选未开仓）")

    return {
        "trades_filtered": t_sel,
        "all_trades_filtered": a_sel,
        "inter_keys": inter_keys,
        "diff_trades": diff_trades
    }

# 使用示例：
# 仅下界 + 时间段
analyze_trades_aligned("trades_ledger.csv", "all_signals_trades.csv",
                        pred_lower=0.09, pred_upper=0.13,
                        start_date=None, end_date=None)
#
# 区间 + 时间段
# analyze_trades_aligned("trades_ledger.csv", "all_signals_trades.csv",
#                        pred_lower=0.05, pred_upper=0.2,
#                        start_date="2023-06-01", end_date="2025-01-31")


[Trades Ledger] 条件(0.09 < predicted_return < 0.13)
总数=1827, pnl均值=0.0696, 盈利数=1146, 盈利占比=62.73%
按 strategy：
          count   avg_pnl  win_count  win_rate
strategy                                      
S1           80  0.317912         64  0.800000
S2          518  0.115361        305  0.588803
S3         1229  0.034120        777  0.632221

[All Signals Trades] 条件(0.09 < predicted_return < 0.13)
总数=3286, pnl均值=0.0475, 盈利数=1899, 盈利占比=57.79%
按 strategy：
          count   avg_pnl  win_count  win_rate
strategy                                      
S1          126  0.200603         60  0.476190
S2          733  0.094318        395  0.538881
S3         2427  0.025370       1444  0.594973

相同记录数: 1827, 不同记录数(候选未开仓): 1459  条件(0.09 < predicted_return < 0.13)

[差集 All_only（候选未开仓）] 条件(0.09 < predicted_return < 0.13)
总数=1459, pnl均值=0.0242, 盈利数=781, 盈利占比=53.53%
按 strategy：
          count   avg_pnl  win_count  win_rate
strategy                                      
S1           46  0.136586      

{'trades_filtered':         symbol strategy entry_date  entry_price   exit_date  exit_price  \
 651         23       S3 2024-06-07     1.720000  2024-06-26    2.020000   
 834         29       S2 2024-08-30    11.060000  2024-11-18   16.570000   
 841         29       S2 2025-09-25    29.850000  2025-10-17   29.000000   
 907         32       S3 2020-10-29    19.732210  2020-11-23   20.994211   
 953         34       S3 2020-05-27    21.279471  2020-06-17   22.359886   
 ...        ...      ...        ...          ...         ...         ...   
 186187  688789       S3 2022-04-26    68.119200  2022-05-11   79.440611   
 186210  688793       S3 2022-01-19    62.705162  2022-02-07   55.838600   
 186228  688798       S3 2024-02-05    44.746425  2024-02-27   56.668861   
 186324  688981       S3 2020-12-22    53.990000  2021-01-04   58.370000   
 186325  688981       S3 2021-03-25    53.250000  2021-04-02   57.210000   
 
          pnl_pct  entry_pos     exit_reason  initial_stop  stop_on

## 持股时间和月均持股数

In [4]:
import pandas as pd
import numpy as np
from typing import Optional

def analyze_holding_periods(
    trades_path: str,
    pred_lower: Optional[float] = None,   # 下界
    pred_upper: Optional[float] = None,   # 上界
    start_date: Optional[str] = None,     # 时间起点，格式 "YYYY-MM-DD"
    end_date: Optional[str] = None        # 时间终点
):
    """
    从 trades_ledger 计算：
      1) 平均持仓天数
      2) 持仓天数分布（直方图）
      3) 月均持股数（按日活跃持仓 → 月平均）

    支持：
      - 预测值过滤（predicted_return）
      - 时间区间过滤（entry_date 在 [start_date, end_date] 内）
    """
    trades = pd.read_csv(trades_path, parse_dates=["entry_date", "exit_date"])

    # ---------- 应用预测阈值 ----------
    if "predicted_return" in trades.columns:
        s = trades["predicted_return"]
        if pred_lower is not None and pred_upper is not None:
            trades = trades[(s > pred_lower) & (s < pred_upper)]
            cond_desc = f"({pred_lower} < predicted_return < {pred_upper})"
        elif pred_lower is not None:
            trades = trades[s > pred_lower]
            cond_desc = f"(predicted_return > {pred_lower})"
        elif pred_upper is not None:
            trades = trades[s < pred_upper]
            cond_desc = f"(predicted_return < {pred_upper})"
        else:
            cond_desc = "(no predicted_return filter)"
    else:
        cond_desc = "(no predicted_return column)"

    # ---------- 应用时间区间 ----------
    if start_date:
        trades = trades[trades["entry_date"] >= pd.to_datetime(start_date)]
    if end_date:
        trades = trades[trades["entry_date"] <= pd.to_datetime(end_date)]
    if start_date or end_date:
        cond_desc += f", 时间: {start_date or '-∞'} → {end_date or '∞'}"

    if trades.empty:
        print(f"⚠️ 没有符合条件的交易 {cond_desc}")
        return None

    # ---------- 1) 计算持仓天数 ----------
    trades["holding_days"] = (trades["exit_date"] - trades["entry_date"]).dt.days
    avg_holding = trades["holding_days"].mean()
    print(f"\n平均持仓时间: {avg_holding:.2f} 天  条件={cond_desc}")

    # ---------- 2) 持仓天数分布 ----------
    dist = trades["holding_days"].value_counts().sort_index()
    print("\n持仓天数分布（前20档）:")
    print(dist.head(20))

    # ---------- 3) 月均持股数 ----------
    records = []
    for _, row in trades.iterrows():
        days = pd.date_range(row["entry_date"], row["exit_date"])
        for d in days:
            records.append((d, row["code"], row["strategy"]))
    pos_df = pd.DataFrame(records, columns=["date", "code", "strategy"])

    daily_holdings = pos_df.groupby("date")["code"].nunique()
    monthly_avg_holdings = daily_holdings.resample("M").mean()
    overall_monthly_avg = monthly_avg_holdings.mean()

    print(f"\n整体月均持股数: {overall_monthly_avg:.2f}")
    print("按月分布：")
    print(monthly_avg_holdings)

    return {
        "trades": trades,
        "holding_days_dist": dist,
        "daily_holdings": daily_holdings,
        "monthly_avg_holdings": monthly_avg_holdings
    }

# 示例：
analyze_holding_periods("trades_ledger.csv", pred_lower=0.1, pred_upper=0.11, start_date=None, end_date=None)


平均持仓时间: 18.59 天  条件=(0.1 < predicted_return < 0.11)

持仓天数分布（前20档）:
holding_days
2     11
3     11
4     27
5     22
6     31
7     26
8     16
9     18
10     9
11    13
12    10
13    14
14    29
15    18
16    12
17    11
18    21
19    14
20     6
21    14
Name: count, dtype: int64

整体月均持股数: 4.83
按月分布：
date
2020-02-29     1.500000
2020-03-31     4.935484
2020-04-30     2.566667
2020-05-31     5.709677
2020-06-30     2.655172
                ...    
2025-06-30     3.633333
2025-07-31    11.258065
2025-08-31     7.838710
2025-09-30     5.066667
2025-10-31     1.714286
Freq: ME, Name: code, Length: 69, dtype: float64


/var/folders/ks/wtspnlt547b0w8gp7ybh7xjr0000gn/T/ipykernel_32388/3349431832.py:72: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_avg_holdings = daily_holdings.resample("M").mean()


{'trades':         symbol strategy entry_date  entry_price  exit_date  exit_price  \
 818         29       S2 2024-08-30    11.060000 2024-11-18   16.570000   
 888         32       S3 2020-10-29    19.732210 2020-11-23   20.994211   
 929         34       S3 2020-05-27    21.279471 2020-06-17   22.359886   
 1832        70       S2 2025-08-07     8.510000 2025-09-04   10.350000   
 2385       301       S3 2021-11-05    20.664557 2021-11-23   23.478041   
 ...        ...      ...        ...          ...        ...         ...   
 179603  688776       S3 2022-01-20   115.302825 2022-02-23  114.702216   
 179626  688777       S3 2021-02-04    52.583760 2021-03-02   58.854373   
 179683  688786       S3 2022-02-22    24.450705 2022-03-02   25.820666   
 179706  688786       S3 2025-04-07    18.037525 2025-04-30   19.647281   
 179886  688981       S3 2020-12-22    53.990000 2021-01-04   58.370000   
 
          pnl_pct  entry_pos     exit_reason  initial_stop  stop_on_exit  \
 818     0.4